# Build 2025-06 (J25) FCC Master File

**Goal:** Combine 5 state-level J25 FiberToThePremises CSVs into a single file matching the existing `all_states_block_level_availability_YYYYMM.csv` schema.

**Inputs:**
- 5 raw J25 fiber CSVs (CA, GA, IL, NY, TX)
- Provider list for holding_company mapping

**Output:** `all_states_block_level_availability_202506.csv`

In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/yugeshchandraroy/Downloads/Projects/market-demographics-fcc-pipeline/data'
OUTPUT_DIR = '/Users/yugeshchandraroy/Downloads/Projects/market-demographics-fcc-pipeline/output'

RAW_FILES = {
    'CA': f'{DATA_DIR}/CA/bdc_06_FibertothePremises_fixed_broadband_J25_03mar2026.csv',
    'GA': f'{DATA_DIR}/GA/bdc_13_FibertothePremises_fixed_broadband_J25_03mar2026.csv',
    'IL': f'{DATA_DIR}/IL/bdc_17_FibertothePremises_fixed_broadband_J25_03mar2026.csv',
    'NY': f'{DATA_DIR}/NY/bdc_36_FibertothePremises_fixed_broadband_J25_03mar2026.csv',
    'TX': f'{DATA_DIR}/TX/bdc_48_FibertothePremises_fixed_broadband_J25_03mar2026.csv',
}

PROVIDER_LIST = f'{DATA_DIR}/bdc_us_provider_list_J25_03mar2026.csv'

print('Build 2025-06 FCC Master File')
print('=' * 40)

Build 2025-06 FCC Master File


## Step 1: Load Provider List

In [2]:
# Load provider list and deduplicate by provider_id, keeping first holding_company
providers = pd.read_csv(PROVIDER_LIST, dtype={'provider_id': str, 'frn': str})
print(f'Provider list raw: {len(providers):,} rows')

providers = providers.drop_duplicates(subset='provider_id', keep='first')[['provider_id', 'holding_company']]
print(f'Provider list deduplicated: {len(providers):,} unique provider_ids')
print(providers.head())

Provider list raw: 2,883 rows
Provider list deduplicated: 2,179 unique provider_ids
  provider_id                               holding_company
0      240058                        Mid-Hudson Cablevision
1      240068                            Neu Ventures, Inc.
2      131087  Rainbow Telecommunications Association, Inc.
4      131060                      TPC ACC Acquisition, LLC
5      240025                Community Antenna System, Inc.


## Step 2: Load and Transform Raw J25 Files

In [3]:
frames = []

for state, filepath in RAW_FILES.items():
    print(f'Loading {state}...')
    df = pd.read_csv(
        filepath,
        dtype={'frn': str, 'provider_id': str, 'block_geoid': str, 'technology': int},
        usecols=['frn', 'provider_id', 'brand_name', 'location_id', 'technology',
                 'max_advertised_download_speed', 'max_advertised_upload_speed',
                 'low_latency', 'business_residential_code', 'state_usps', 'block_geoid']
    )
    print(f'  Rows: {len(df):,}  |  Unique blocks: {df["block_geoid"].nunique():,}  |  Tech codes: {sorted(df["technology"].unique())}')
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f'\nCombined raw: {len(raw):,} rows  |  {raw["block_geoid"].nunique():,} unique blocks')
print(f'Technology codes: {sorted(raw["technology"].unique())}')

Loading CA...
  Rows: 5,113,420  |  Unique blocks: 217,223  |  Tech codes: [np.int64(50)]
Loading GA...
  Rows: 3,280,132  |  Unique blocks: 134,482  |  Tech codes: [np.int64(50)]
Loading IL...
  Rows: 2,374,355  |  Unique blocks: 147,545  |  Tech codes: [np.int64(50)]
Loading NY...
  Rows: 6,464,753  |  Unique blocks: 180,286  |  Tech codes: [np.int64(50)]
Loading TX...
  Rows: 9,620,176  |  Unique blocks: 354,286  |  Tech codes: [np.int64(50)]

Combined raw: 26,852,836 rows  |  1,033,822 unique blocks
Technology codes: [np.int64(50)]


## Step 3: Map to Processed Schema

In [4]:
# Derive FIPS codes from block_geoid
raw['state_fips'] = raw['block_geoid'].str[:2]
raw['county_fips'] = raw['block_geoid'].str[:5]
raw['tract_fips'] = raw['block_geoid'].str[:11]

# Join with provider list for holding_company
raw = raw.merge(providers, on='provider_id', how='left')
missing_hc = raw['holding_company'].isna().sum()
print(f'Holding company: {missing_hc:,} rows missing ({100*missing_hc/len(raw):.2f}%)')

# Technology name (all records are tech 50)
raw['technology_name'] = 'Fiber to the Premises (FTTP)'

# Rename speed columns
raw['max_download'] = raw['max_advertised_download_speed']
raw['max_upload'] = raw['max_advertised_upload_speed']

# Residential / Business flags from business_residential_code
raw['serves_residential'] = raw['business_residential_code'].isin(['R', 'X']).astype(int)
raw['serves_business'] = raw['business_residential_code'].isin(['B', 'X']).astype(int)

# Filing period
raw['filing_period'] = '2025-06'

# Select final columns in target schema order
output_cols = [
    'block_geoid', 'state_fips', 'county_fips', 'tract_fips',
    'provider_id', 'holding_company', 'technology', 'technology_name',
    'max_download', 'max_upload', 'serves_residential', 'serves_business',
    'low_latency', 'filing_period'
]

result = raw[output_cols].copy()
print(f'\nProcessed schema: {result.shape}')
print(f'Columns: {list(result.columns)}')
print(result.head())

Holding company: 0 rows missing (0.00%)

Processed schema: (26852836, 14)
Columns: ['block_geoid', 'state_fips', 'county_fips', 'tract_fips', 'provider_id', 'holding_company', 'technology', 'technology_name', 'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency', 'filing_period']
       block_geoid state_fips county_fips   tract_fips provider_id  \
0  060190082007019         06       06019  06019008200      131173   
1  060610209083005         06       06061  06061020908      130335   
2  060610211091015         06       06061  06061021109      130335   
3  060610210431018         06       06061  06061021043      130335   
4  060610210341006         06       06061  06061021034      130335   

         holding_company  technology               technology_name  \
0  Sebastian Enterprises          50  Fiber to the Premises (FTTP)   
1   Fidium Holdings, LLC          50  Fiber to the Premises (FTTP)   
2   Fidium Holdings, LLC          50  Fiber to the Premis

## Step 4: Validate

In [5]:
print('Validation')
print('=' * 50)

# Check no nulls in critical columns
critical = ['block_geoid', 'state_fips', 'county_fips', 'tract_fips',
            'provider_id', 'technology', 'max_download', 'max_upload', 'filing_period']
nulls = result[critical].isnull().sum()
print(f'Null counts in critical columns:\n{nulls[nulls > 0]}')
if nulls.sum() == 0:
    print('  No nulls in critical columns.')

# Technology should all be 50
assert (result['technology'] == 50).all(), 'Non-fiber technology codes found!'
print(f'Technology: all records are code 50 (FTTP)')

# Filing period should all be 2025-06
assert (result['filing_period'] == '2025-06').all()
print(f'Filing period: all 2025-06')

# State breakdown
print(f'\nState breakdown:')
for state_fips, group in result.groupby('state_fips'):
    print(f'  {state_fips}: {len(group):,} rows, {group["block_geoid"].nunique():,} unique blocks')

print(f'\nTotal: {len(result):,} rows, {result["block_geoid"].nunique():,} unique blocks')
print(f'Unique providers: {result["provider_id"].nunique():,}')

Validation
Null counts in critical columns:
Series([], dtype: int64)
  No nulls in critical columns.
Technology: all records are code 50 (FTTP)
Filing period: all 2025-06

State breakdown:
  06: 5,113,420 rows, 217,223 unique blocks
  13: 3,280,132 rows, 134,482 unique blocks
  17: 2,374,355 rows, 147,545 unique blocks
  36: 6,464,753 rows, 180,286 unique blocks
  48: 9,620,176 rows, 354,286 unique blocks

Total: 26,852,836 rows, 1,033,822 unique blocks
Unique providers: 368


## Step 5: Export

In [6]:
output_path = f'{OUTPUT_DIR}/all_states_block_level_availability_202506.csv'
result.to_csv(output_path, index=False)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f'Exported: {output_path}')
print(f'  Rows: {len(result):,}')
print(f'  Columns: {len(result.columns)}')
print(f'  File size: {file_size_mb:.1f} MB')
print(f'\nNotebook 02 complete!')

Exported: /Users/yugeshchandraroy/Downloads/Projects/market-demographics-fcc-pipeline/output/all_states_block_level_availability_202506.csv
  Rows: 26,852,836
  Columns: 14
  File size: 3103.5 MB

Notebook 02 complete!
